# 01 Data Cleaning and Summary Generation

This notebook documents how the project moves from the raw ACLED export to the processed event-level CSV and the summary CSV files used by the report.

Outputs generated here:

- `data/acled_mena_processed.csv`
- `data/yearly_shift_summary.csv`
- `data/spatial_bucket_summary.csv`
- `data/country_pattern_summary.csv`


In [ ]:
from pathlib import Path
import sys

import pandas as pd

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "code" / "project_utils.py").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Could not find the project root folder.")


ROOT_DIR = find_project_root(Path.cwd())

CODE_DIR = ROOT_DIR / "code"
DATA_DIR = ROOT_DIR / "data"
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from project_utils import (
    prepare_analysis_data,
    summarize_country_patterns,
    summarize_spatial_bucket,
    summarize_yearly,
)

RAW_DATA_PATH = DATA_DIR / "ACLED Data_MENA_Raw.csv"
PROCESSED_PATH = DATA_DIR / "acled_mena_processed.csv"
YEARLY_SUMMARY_PATH = DATA_DIR / "yearly_shift_summary.csv"
SPATIAL_SUMMARY_PATH = DATA_DIR / "spatial_bucket_summary.csv"
COUNTRY_SUMMARY_PATH = DATA_DIR / "country_pattern_summary.csv"

RAW_DATA_PATH


## Step 1: Load, filter, and classify events

`prepare_analysis_data()` performs the shared cleaning logic:

- loads the raw ACLED CSV,
- keeps `Middle East` and `Northern Africa`,
- drops rows missing core fields,
- parses dates,
- creates capital-area and remote-violence indicators,
- calculates distance to international land borders using Natural Earth boundaries,
- creates border-proximate and spatial-bucket variables.


In [ ]:
processed = prepare_analysis_data()
processed.to_csv(PROCESSED_PATH, index=False)

processed.shape, PROCESSED_PATH


## Step 2: Create the report sample

The processed event-level file keeps the cleaned event records. The final report's analytical sample excludes 2025 and uses 1997-2024, matching the text and published webpage.


In [ ]:
report_df = processed[processed["year"].between(1997, 2024)].copy()

{
    "processed_rows": len(processed),
    "report_rows_1997_2024": len(report_df),
    "min_year": int(report_df["year"].min()),
    "max_year": int(report_df["year"].max()),
}


## Step 3: Write summary CSV files

These summaries support the report figures and aggregate claims.


In [ ]:
yearly = summarize_yearly(report_df)
spatial = summarize_spatial_bucket(report_df)
country = summarize_country_patterns(report_df)

yearly.to_csv(YEARLY_SUMMARY_PATH, index=False)
spatial.to_csv(SPATIAL_SUMMARY_PATH, index=False)
country.to_csv(COUNTRY_SUMMARY_PATH, index=False)

[YEARLY_SUMMARY_PATH, SPATIAL_SUMMARY_PATH, COUNTRY_SUMMARY_PATH]


## Step 4: Sanity checks

These checks reproduce the headline counts cited in the report.


In [ ]:
checks = {
    "all_events_1997_2024": len(report_df),
    "border_proximate_events": int(report_df["is_border_proximate"].sum()),
    "capital_area_events": int(report_df["is_capital_area"].sum()),
    "overlap_events": int((report_df["is_border_proximate"] & report_df["is_capital_area"]).sum()),
    "noncapital_border_proximate_events": int((report_df["is_border_proximate"] & ~report_df["is_capital_area"]).sum()),
    "capital_remote_events": int(report_df["capital_remote_event"].sum()),
}

checks


In [ ]:
yearly.tail()
